# Lekcija 05 - Agentic RAG


## Postavljanje

Ovaj bilježnik prikazuje Agentic RAG (Retrieval-Augmented Generation) obrazac koristeći Microsoft Agent Framework.

**Preduvjeti:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — vaš Azure AI Search endpoint
- `AZURE_SEARCH_API_KEY` — vaš Azure AI Search API ključ
- Azure OpenAI implementiran putem varijabli okoline
- Azure CLI prijavljen (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Što je Agentic RAG?

Tradicionalni RAG slijedi fiksni tijek rada: dohvatiti dokumente, zatim generirati odgovor. **Agentic RAG** ide korak dalje dajući agentu autonomiju da odluči **kada** i **kako** dohvatiti informacije.

S Agentic RAG-om, agent može:
- **Odlučiti** je li dohvat potreban prije nego što odgovori na pitanje
- **Odabrati** koji izvor podataka ili alat treba upitati
- **Procijeniti** dohvaćene rezultate i obaviti dodatne dohvate ako prvi pokušaj nije dovoljan
- **Kombinirati** informacije iz više koraka dohvaćanja u koherentan odgovor

To čini agenta fleksibilnijim i preciznijim u usporedbi sa statičnim tijekom rada dohvati, zatim generiraj.


## Stvaranje alata za pretraživanje

U Agentic RAG-u, vanjski izvori podataka su umotani kao **alatke** koje agent može pozvati po potrebi. To agentu omogućuje da tretira dohvaćanje kao još jednu radnju koju može poduzeti, umjesto kao obavezni korak.

Ispod definiramo bazu znanja o putovanjima i izlažemo je kao alatku koju agent može pozvati za pronalaženje informacija o destinaciji.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Izgradnja RAG agenta

Sada stvaramo agenta kojem je naređeno da **uvijek prvo pretraži informacije prije odgovora**. Agent koristi alat `search_travel_knowledge` kako bi svoje odgovore utemeljio na bazi znanja, a ne na vlastitim podacima za obuku.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterativno dohvaćanje — obrazac Maker-Checker

Ključna prednost Agentic RAG-a je **iterativno dohvaćanje**. Agent može obaviti više krugova pretraživanja kako bi provjerio, doradio ili proširio svoja početna saznanja — slično radnom procesu "maker-checker":

1. **Korak maker**: Agent dohvaća početne informacije i sastavlja odgovor.
2. **Korak checker**: Agent obavlja dodatna dohvaćanja radi provjere detalja ili popunjavanja praznina.

Ispod je agentu postavljeno pitanje koje zahtijeva usporedbu više destinacija, potičući ga na višestruko pretraživanje.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Sažetak

U ovoj lekciji naučili ste kako izgraditi **Agentic RAG** sustav koristeći Microsoft Agent Framework:

- **Agentic RAG** omogućuje agentima da samostalno odlučuju kada dohvatiti informacije, čineći dohvat dinamičnim umjesto fiksnim.
- **Alati kao izvori podataka**: Vanjske baze znanja (kao što je Azure AI Search) umotane su kao alati koje agent može pozivati.
- **Iterativni dohvat**: obrazac maker-checker omogućuje agentu da obavi više rundi dohvaćanja — pretraživanje, provjeru i doradu — prije nego što proizvede konačni odgovor.

U produkciji biste zamijenili memorijski `TRAVEL_KNOWLEDGE_BASE` pravim Azure AI Search indeksom za rukovanje dohvaćanjem velikih količina putnih dokumenata.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
